In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "validation").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from validation.notebook_bootstrap import bootstrap

bedrock_model_arn, load_workshop_state, persist_workshop_state_file = bootstrap()


# Amazon Bedrock Knowledge Bases - Exemplo completo (end to end) usando múltiplos data source(s)

Este notebook fornece um código de exemplo para construir um exemplo end-to-end de uma aplicação RAG usando Amazon Bedrock Knowledge Bases e ingerir documentos no índice a partir de várias data sources (S3, Confluence, Sharepoint, Salesforce e Web). Observe que você pode adicionar até 5 data sources.


#### Percurso do notebook

Um pipeline de dados que ingere documentos (normalmente armazenados em múltiplas data sources) em uma knowledge base, ou seja, um banco de dados vetorial como o Amazon OpenSearch Service Serverless (AOSS), para que fiquem disponíveis para consulta quando uma pergunta é recebida.

- Carregar os documentos na knowledge base conectando várias data sources (S3, Confluence, Sharepoint, Salesforce e Web).
- Ingestão (Ingestion) - a Knowledge Base vai dividi-los em chunks menores (com base na estratégia selecionada), gerar embeddings e armazená-los no vector store associado.

<!-- ![data_ingestion.png](./images/data_ingestion.png) -->
<img src="./images/data_ingestion.png" width=50% height=20% />


#### Passos:
- Criar a execution role da Knowledge Base com as políticas necessárias para acessar dados de várias data sources (S3, Confluence, Sharepoint, Salesforce e Web) e escrever embeddings no OSS.
- Criar um índice OpenSearch Serverless vazio.
- Pré-requisito:
    - Para S3, criar o bucket s3 (se não existir) e fazer upload dos dados
    - Para as demais data sources - consulte os pré-requisitos na [página de documentação da AWS](https://docs.aws.amazon.com/bedrock/latest/userguide/data-source-connectors.html) correspondente
- Criar a knowledge base
- Criar a(s) data source(s) dentro da knowledge base
- Para cada data source, iniciar os ingestion jobs usando as APIs da KB, que vão ler os dados da data source, fazer o chunking, converter os chunks em embeddings usando o modelo Amazon Titan Embeddings e então armazenar esses embeddings no AOSS. Tudo isso sem precisar construir, implantar e gerenciar o pipeline de dados.

Uma vez que os dados estejam disponíveis na Bedrock Knowledge Base, uma aplicação de question answering pode ser construída usando as APIs de Knowledge Base fornecidas pelo Amazon Bedrock.


<div class="alert alert-block alert-info">
<b>Nota:</b> Certifique-se de habilitar o acesso aos modelos `amazon.nova-micro-v1:0`, `us.anthropic.claude-haiku-4-5-20251001-v1:0` (modelo de geração de texto padrão, configurável via `BEDROCK_TEXT_MODEL_ID`), `amazon.titan-text-express-v1` e `Titan Text Embeddings V2` no console do Amazon Bedrock.
<br> -------------------------------------------------------------------------------------------------------------------------------------------------------   </br>

Execute o notebook célula por célula em vez de usar a opção "Run All Cells".
</div>

## Setup
Antes de executar o restante deste notebook, você precisará executar as células abaixo para (garantir que as bibliotecas necessárias estejam instaladas e) conectar-se ao Bedrock.

In [ ]:
%pip install --upgrade pip --quiet
%pip install -r ../requirements.txt --no-deps --quiet
%pip install -r ../requirements.txt --upgrade --quiet

In [ ]:
# Kernel restart is intentionally skipped in corrected notebooks.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os
import sys
import time
import boto3
import logging
import pprint
import json

# Set the path to import module
from pathlib import Path
current_path = Path().resolve()
current_path = current_path.parent
if str(current_path) not in sys.path:
    sys.path.append(str(current_path))
# Print sys.path to verify
# print(sys.path)

from utils.knowledge_base import BedrockKnowledgeBase

In [ ]:
#Clients
s3_client = boto3.client('s3')
sts_client = boto3.client('sts')
session = boto3.session.Session()
region =  session.region_name
account_id = sts_client.get_caller_identity()["Account"]
bedrock_agent_client = boto3.client('bedrock-agent')
bedrock_agent_runtime_client = boto3.client('bedrock-agent-runtime') 
logging.basicConfig(format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)
region, account_id

In [ ]:
import time
import uuid

# Use a per-run token so every derived AWS resource name is globally unique.
run_token = uuid.uuid4().hex[:10]
suffix = f"{run_token}-f"

knowledge_base_name = f"bedrock-sample-knowledge-base-{run_token}"
knowledge_base_description = "Multi data source knowledge base."

bucket_name = f"bedrock-kb-{account_id}-{run_token}"
intermediate_bucket_name = f"{bucket_name}-intermediate"
foundation_model = os.getenv("BEDROCK_TEXT_MODEL_ID", "us.anthropic.claude-haiku-4-5-20251001-v1:0")


In [ ]:
print(boto3.__version__)

### Agora você pode adicionar múltiplas e diferentes data sources (S3, Confluence, Sharepoint, Salesforce, Web Crawler) a uma Knowledge Base. Neste notebook, vamos testar a criação de uma Knowledge Base com múltiplas e diferentes data sources.

Cada data source pode ter pré-requisitos diferentes; consulte a documentação da AWS para mais informações.

In [ ]:
# For this notebook, we'll create Knowledge Base with multiple data sources ( 1 S3 bucket, 1 confluence page, 1 Sharepoint site, 1 Salesforce site, 1 Web Crawler)

data_bucket_name = f'{bucket_name}-1' # replace it with your first bucket name.

## Below is a list of data sources including, 1 S3 buckets, 1 confluence, 1 Sharepoint, 1 Salesforce connectors
## Please uncomment the data sources that you want to add and update the placeholder values accordingly.

data_sources=[
                {"type": "S3", "bucket_name": data_bucket_name}, 
                
                # {"type": "CONFLUENCE", "hostUrl": "https://example.atlassian.net", "authType": "BASIC",
                #  "credentialsSecretArn": f"arn:aws::secretsmanager:{region_name}:secret:<<your_secret_name>>"},

                # {"type": "SHAREPOINT", "tenantId": "888d0b57-69f1-4fb8-957f-e1f0bedf64de", "domain": "yourdomain",
                #   "authType": "OAUTH2_CLIENT_CREDENTIALS",
                #  "credentialsSecretArn": f"arn:aws::secretsmanager:{region_name}:secret:<<your_secret_name>>",
                #  "siteUrls": ["https://yourdomain.sharepoint.com/sites/mysite"]
                # },

                # {"type": "SALESFORCE", "hostUrl": "https://company.salesforce.com/", "authType": "OAUTH2_CLIENT_CREDENTIALS",
                #  "credentialsSecretArn": f"arn:aws::secretsmanager:{region_name}:secret:<<your_secret_name>>"
                # },

                # {"type": "WEB", "seedUrls": [{ "url": "https://www.examplesite.com"}],
                #  "inclusionFilters": ["https://www\.examplesite\.com/.*\.html"],
                #  "exclusionFilters": ["https://www\.examplesite\.com/contact-us\.html"]
                # }
            ]
                
pp = pprint.PrettyPrinter(indent=2)

## Criar a Knowledge Base

In [ ]:
knowledge_base = BedrockKnowledgeBase(
    kb_name=f'{knowledge_base_name}',
    kb_description=knowledge_base_description,
    data_sources=data_sources,
    chunking_strategy = "FIXED_SIZE",
    suffix = suffix
)


def persist_workshop_state(ingestion_started=False):
    """Persist enough information for dependent notebooks and cleanup."""
    kb_summary = knowledge_base.knowledge_base
    data_source_summaries = knowledge_base.data_source
    workshop_state = {
        "schema_version": 1,
        "kb_id": kb_summary["knowledgeBaseId"],
        "knowledge_base_name": kb_summary["name"],
        "region": region,
        "account_id": account_id,
        "data_source_ids": [item["dataSourceId"] for item in data_source_summaries],
        "data_source_names": [item.get("name") for item in data_source_summaries],
        "s3_bucket_names": list(knowledge_base.bucket_names),
        "resource_suffix": suffix,
        "cleanup_required": True,
        "ingestion_started": ingestion_started,
        "persisted_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "resource_names": {
            "knowledge_base": knowledge_base.kb_name,
            "data_sources": [item.get("name") for item in data_source_summaries],
            "s3_buckets": list(knowledge_base.bucket_names),
            "vector_store": getattr(knowledge_base, "vector_store_name", None),
            "vector_index": getattr(knowledge_base, "index_name", None),
            "iam_role": getattr(knowledge_base, "kb_execution_role_name", None),
        },
    }
    ip = get_ipython()
    if "store" not in ip.magics_manager.magics["line"]:
        ip.run_line_magic("load_ext", "storemagic")
    ip.user_ns["workshop_state"] = workshop_state
    persist_workshop_state_file(workshop_state)
    return workshop_state


workshop_state = persist_workshop_state()
print(f"Persisted workshop_state for Knowledge Base {workshop_state['kb_id']} before ingestion.")


### Baixar os dados para ingerir em nossa knowledge base.
Vamos usar os seguintes dados:
 - dados sintéticos armazenados em um diretório local, como primeira data source

#### Fazer upload dos dados para a data source do bucket S3

In [ ]:
def upload_directory(path, bucket_name):
        for root,dirs,files in os.walk(path):
            for file in files:
                file_to_upload = os.path.join(root,file)
                print(f"uploading file {file_to_upload} to {bucket_name}")
                s3_client.upload_file(file_to_upload,bucket_name,file)

upload_directory("../synthetic_dataset", data_bucket_name)

### Iniciar o ingestion job
Depois que a KB e a(s) data source(s) forem criadas, podemos iniciar o ingestion job para cada data source.
Durante o ingestion job, a KB vai buscar os documentos na data source, pré-processá-los para extrair o texto, dividi-los (chunk) com base no tamanho de chunk fornecido, criar embeddings de cada chunk e então escrever no banco de dados vetorial, neste caso o OSS.

NOTA: Atualmente, você só pode iniciar um ingestion job por vez.

In [ ]:
# ensure that the kb is available
time.sleep(30)
# sync knowledge base
knowledge_base.start_ingestion_job()

In [ ]:
# Refresh the shared state after the ingestion job has been started.
kb_id = knowledge_base.get_knowledge_base_id()
workshop_state = persist_workshop_state(ingestion_started=True)
print(f"Persisted workshop_state for Knowledge Base {kb_id}; run cleanup only after reviewing this state.")


### 2.2 Testar a Knowledge Base
Agora que a Knowledge Base está disponível, podemos testá-la usando as funções [**retrieve**](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent-runtime/client/retrieve.html) e [**retrieve_and_generate**](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent-runtime/client/retrieve_and_generate.html).

#### Testando a Knowledge Base com a API Retrieve and Generate

Vamos primeiro testar a knowledge base usando a API retrieve and generate. Com essa API, o Bedrock cuida de recuperar as referências necessárias da knowledge base e gerar a resposta final usando um foundation model do Bedrock.

query = `Provide a summary of consolidated statements of cash flows of Octank Financial for the fiscal years ended December 31, 2019.`

A resposta correta para essa query, de acordo com o par de QA ground truth, é:
```
The cash flow statement for Octank Financial in the year ended December 31, 2019 reveals the following:
- Cash generated from operating activities amounted to $710 million, which can be attributed to a $700 million profit and non-cash charges such as depreciation and amortization.
- Cash outflow from investing activities totaled $240 million, with major expenditures being the acquisition of property, plant, and equipment ($200 million) and marketable securities ($60 million), partially offset by the sale of property, plant, and equipment ($40 million) and maturing marketable securities ($20 million).
- Financing activities resulted in a cash inflow of $350 million, stemming from the issuance of common stock ($200 million) and long-term debt ($300 million), while common stock repurchases ($50 million) and long-term debt payments ($100 million) reduced the cash inflow.
Overall, Octank Financial experienced a net cash enhancement of $120 million in 2019, bringing their total cash and cash equivalents to $210 million.
```

In [ ]:
query = "Provide a summary of consolidated statements of cash flows of Octank Financial for the fiscal years ended December 31, 2019?"

In [ ]:
foundation_model = os.getenv("BEDROCK_TEXT_MODEL_ID", "us.anthropic.claude-haiku-4-5-20251001-v1:0")

response = bedrock_agent_runtime_client.retrieve_and_generate(
    input={
        "text": query
    },
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            'knowledgeBaseId': kb_id,
            "modelArn": bedrock_model_arn(foundation_model, region),
            "retrievalConfiguration": {
                "vectorSearchConfiguration": {
                    "numberOfResults":5
                } 
            }
        }
    }
)

print(response['output']['text'],end='\n'*2)

Como você pode ver, com a API retrieve and generate obtemos a resposta final diretamente e não vemos as diferentes fontes usadas para gerar essa resposta. Vamos agora recuperar as informações de origem da knowledge base com a API retrieve.

#### Testando a Knowledge Base com a API Retrieve
Se você precisa de uma camada extra de controle, pode recuperar os chunks que melhor correspondem à sua query usando a API retrieve. Nesta configuração, podemos definir o número desejado de resultados e controlar a resposta final com a lógica da sua própria aplicação. A API então fornece o conteúdo correspondente, sua localização no S3, o similarity score e o metadata do chunk.

In [ ]:
response_ret = bedrock_agent_runtime_client.retrieve(
    knowledgeBaseId=kb_id, 
    nextToken='string',
    retrievalConfiguration={
        "vectorSearchConfiguration": {
            "numberOfResults":5,
        } 
    },
    retrievalQuery={
        "text": "How many new positions were opened across Amazon's fulfillment and delivery network?"
    }
)

def response_print(retrieve_resp):
#structure 'retrievalResults': list of contents. Each list has content, location, score, metadata
    for num,chunk in enumerate(response_ret['retrievalResults'],1):
        print(f'Chunk {num}: ',chunk['content']['text'],end='\n'*2)
        print(f'Chunk {num} Location: ',chunk['location'],end='\n'*2)
        print(f'Chunk {num} Score: ',chunk['score'],end='\n'*2)
        print(f'Chunk {num} Metadata: ',chunk['metadata'],end='\n'*2)

response_print(response_ret)

### Limpeza (Clean up)
Certifique-se de descomentar e executar a seção abaixo para excluir todos os recursos.

In [ ]:
# Cleanup is intentionally deferred to full_cleanup.ipynb.
print("Cleanup deferred to full_cleanup.ipynb.")
